# Data Source
This project uses two primary datasets:


*   Fangraphs play statistics (2015-2019): Containing player statistics for the first half of the season (Before the all-star break). These features serve as input data for the model.
*   All-Star rosters (2015–2019): A manually compiled dataset identifying players selected to the MLB All-Star Game for each season. This dataset is used to construct the target variable for classification.

The goal is to combine these datasets to create a labeled dataset where each player-season observation is associated with an All-Star selection outcome.



In [3]:
|import pandas as pd

# Reading in 1st half stats from 2015 - 2019
years = [2015,2016,2017,2018,2019]

player_stats = pd.DataFrame()

for year in years:
  path = f"/content/pos_player_{year}.csv"

  stats = pd.read_csv(path)
  print(len(stats))
  stats['year'] = year

  player_stats = pd.concat([player_stats, stats], ignore_index=True)


# Reading in all-star rosters from 2015-2019 (starters and reserves)
all_stars = pd.read_csv('all_stars.csv')

162
167
166
164
156


# Data Normalization and Join Preparation
To combine player statistics with All-Star selections, both datasets were standardized to allow for matching on (player name, team, year).

Name fields were normalized by applying consistent formatting to reduce mismatches across sources. Team identifiers and season values were also aligned to ensure consistency.

Due to the lack of a shared unique identifier across datasets, this join relies on these normalized fields as a proxy. While this approach may introduce minor mismatches, it is sufficient for constructing a labeled dataset at the player-season level.

Duplicate records were checked and removed to ensure that each player-season combination appears only once prior to merging.


In [4]:
import re
import unicodedata
# Getting rid of discremencies between team names for joining
team_mapping = {
    "KC": "KCR",
    "SF": "SFG",
    "WSH": "WSN",
    "SD": "SDP",
    "TB": "TBR"
}

all_stars['team'] = all_stars['team'].apply(lambda x: team_mapping.get(x, x))

# Normalizing all player names to ensure proper mapping (AI generated)
def normalize_name(name):
    name = unicodedata.normalize('NFKD', str(name)).encode('ASCII', 'ignore').decode()
    name = name.lower().strip()
    name = name.replace('.', '')  # remove all periods
    name = re.sub(r'\s+', ' ', name)  # collapse multiple spaces into one
    name = name.replace(' jr', '').replace(' sr', '')
    name = name.replace(' ii', '').replace(' iii', '')
    while re.search(r'\b[a-z]\s[a-z]\b', name):
      name = re.sub(r'\b([a-z])\s([a-z])\b', r'\1\2', name)
    return name.strip()

print(len(all_stars))
all_stars['player_name'] = all_stars['player_name'].apply(normalize_name)
print(len(all_stars))
print(len(player_stats))
player_stats['Name'] = player_stats['Name'].apply(normalize_name)
print(len(player_stats))
# Joining player
all_stars.rename(columns={'player_name': 'Name', 'team': 'Team'}, inplace=True)
all_stars['all_star'] = 1
final_df = pd.merge(player_stats, all_stars, on=['Name', 'Team', 'year'], how='left')
final_df['all_star'] = final_df['all_star'].fillna(0).astype(int)

228
228
815
815


# Feature Engineering
Feature engineering was kept minimal, as the FanGraphs dataset already provides a rich set of advanced performance metrics that capture key aspects of player value.

These features are widely used as comprehensive indicators of offensive and overall performance, reducing the need for additional transformations or derived variables.

The focus of this project is therefore on evaluating how well standard machine learning models can leverage these existing features, rather than constructing new ones.

In [5]:
# Dropping columns which will not be used within modeling
final_df.drop(columns=['Team', 'Name', 'NameASCII', 'PlayerId', 'MLBAMID', 'BABIP', 'wOBA', 'xwOBA', 'Off'], inplace=True)

# Power + speed feature (helps model capture 2-way players)
final_df['power_speed'] = final_df['HR'] + final_df['SB']

# BB % / K % captures the players plate discipline
final_df['plate_discipline'] = final_df.apply(lambda row: row['BB%'] / row['K%'] if row['K%'] != 0 else row['BB%'], axis = 1)

# Filtering out players with less than 150 PA's
final_df = final_df[final_df['PA'] >= 150]

# Modeling Approach
This project frames All-Star selection as a binary classification problem, where each player-season observation is labeled as either selected (1) or not selected (0).

Several baseline models were evaluated, including Logistic Regression, Random Forest, and Gradient Boosting, to compare performance across both linear and non-linear approaches.

Given the class imbalance inherent in All-Star selection, models were trained using class weighting and evaluated using F1-score to balance precision and recall.

Model performance was assessed using stratified cross-validation on the training set (2015–2018), with final evaluation conducted on a holdout test set (2019) to simulate out-of-sample prediction.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

# Creating the train test split (train on data from 2015-2018)
train_split = final_df[final_df['year'] != 2019]
test_split = final_df[final_df['year'] == 2019]
X_train = train_split.drop(columns=['all_star', 'year'])
y_train = train_split['all_star']
X_test = test_split.drop(columns=['all_star', 'year'])
y_test = test_split['all_star']

# Create numeric feature pipeline
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy='mean')),
    ("scaler", StandardScaler())
])

# Column Transformer piplien
transformer = ColumnTransformer([
    ("num_pipe", numeric_pipeline, X_train.columns)
])

# Pipeline for Logistic Regression Model
logistic_pipeline = Pipeline([
    ("transformer", transformer),
    ("model", LogisticRegression(class_weight='balanced', random_state=42, solver='liblinear'))
])

# Pipeline for Random Forest Classification Model
forest_pipeline = Pipeline([
    ("transformer", transformer),
    ("model", RandomForestClassifier(class_weight='balanced', random_state=42))
])

# Pipeline for Gradient Boost model
boost_pipeline = Pipeline([
    ("transformer", transformer),
    ("model", GradientBoostingClassifier(random_state=42))
])

# Stratified CV for Grid Search because of data inbalance
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic regression parameter tuning
logistic_params = {
    "model__C": [0.5, 1.5, 2.5],
    "model__penalty": ['l2', 'l1']
}
logistic_grid = GridSearchCV(logistic_pipeline, logistic_params, cv=skf, scoring='f1')

# Random Forest Parameter tuning
forest_params = {
    'model__n_estimators': [50,100, 150],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_split': [10, 15, 20]
}
forest_grid = GridSearchCV(forest_pipeline, forest_params, cv = skf, scoring='f1')

# Gradient Boost Parameter tuning
boost_params = {
    'model__n_estimators': [50, 150],
    'model__max_depth': [5, 15],
    'model__learning_rate': [0.05, 0.1, .15]
}
boost_grid = GridSearchCV(boost_pipeline, boost_params, cv = skf, scoring='f1')

# Training all 3 models
logistic_grid.fit(X_train, y_train)
forest_grid.fit(X_train, y_train)
boost_grid.fit(X_train, y_train)

print("Logistic Regression Performance on Training Data\n")
logistic_idx = logistic_grid.best_index_
print(f"Logistic Regression best score: {logistic_grid.best_score_}")
print(f"Best Model mean f1: {logistic_grid.cv_results_['mean_test_score'][logistic_idx]}")
print(f"Best Model std f1: {logistic_grid.cv_results_['std_test_score'][logistic_idx]}\n")

print("Random Forest Performance on Training Data\n")
forest_idx = forest_grid.best_index_
print(f"Random Forest best score: {forest_grid.best_score_}")
print(f"Best Model mean f1: {forest_grid.cv_results_['mean_test_score'][forest_idx]}")
print(f"Best Model std f1: {forest_grid.cv_results_['std_test_score'][forest_idx]}\n")

print("Gradient Boost Performance on Training Data\n")
boost_idx = boost_grid.best_index_
print(f"Gradient Boost best score: {boost_grid.best_score_}")
print(f"Best Model mean f1: {boost_grid.cv_results_['mean_test_score'][boost_idx]}")
print(f"Best Model std f1: {boost_grid.cv_results_['std_test_score'][boost_idx]}\n")



Logistic Regression Performance on Training Data

Logistic Regression best score: 0.7132217782217782
Best Model mean f1: 0.7132217782217782
Best Model std f1: 0.046474502963599194

Random Forest Performance on Training Data

Random Forest best score: 0.7180437919568355
Best Model mean f1: 0.7180437919568355
Best Model std f1: 0.06673014556003042

Gradient Boost Performance on Training Data

Gradient Boost best score: 0.6865100238127468
Best Model mean f1: 0.6865100238127468
Best Model std f1: 0.0633610374155466



# Model Comparison and Selection
Model performance was evaluated using stratified cross-validation on the training data (2015–2018), with F1-score as the primary metric due to class imbalance.

The results are summarized below:


*   Logistic Regression: mean F1 = 0.710, std = 0.046
*   Random Forest: mean F1 = 0.718, std = 0.067
*   Gradient Boosting: mean F1 = 0.687, std = 0.063

Random Forest achieved the highest average F1-score, indicating slightly better overall performance compared to the other models. While the improvement over Logistic Regression is modest, Random Forest was selected due to its ability to capture non-linear relationships and feature interactions.

Given this performance and its flexibility, Random Forest was chosen as the final model for evaluation on the 2019 holdout test set.

In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score,f1_score

best_model = forest_grid.best_estimator_
y_hat = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("Best Model Classification Report\n")
print(f"{classification_report(y_test, y_hat)}\n")

print("Best Model Confusion Matrix")
print(f"{confusion_matrix(y_test, y_hat)}\n")

print("Best Model ROC AUC Score")
print(f"{roc_auc_score(y_test, y_proba)}\n")

print("Best Model F1 Score")
print(f1_score(y_test, y_hat))

Best Model Classification Report

              precision    recall  f1-score   support

           0       0.90      0.84      0.87       117
           1       0.60      0.72      0.65        39

    accuracy                           0.81       156
   macro avg       0.75      0.78      0.76       156
weighted avg       0.82      0.81      0.81       156


Best Model Confusion Matrix
[[98 19]
 [11 28]]

Best Model ROC AUC Score
0.8522901599824676

Best Model F1 Score
0.6511627906976745


# Threshold Analysis
As part of model evaluation, different classification thresholds were tested to analyze the tradeoff between precision and recall.

In a production setting, threshold selection would typically be performed using a separate validation set to avoid biasing results on the test data. However, this step was included here as an exploratory exercise to better understand how threshold adjustments impact model performance.

The results show that lower thresholds increase recall at the expense of precision, while higher thresholds have the opposite effect. In this case, the default threshold of 0.5 provided the best balance, achieving the highest F1-score.

Given that alternative thresholds did not produce a meaningful improvement, the default threshold of 0.5 would be retained for any production use.

In [8]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

y_prob = best_model.predict_proba(X_test)[:,1]

thresholds = np.arange(0.1, .9, 0.05)

scores = {}
best_f1_threshold = (None, 0)

for threshold in thresholds:
  y_hat_thresh = (y_prob >= threshold).astype(int)
  scores[threshold] = {"precision": precision_score(y_test, y_hat_thresh, zero_division=0),
                       "recall": recall_score(y_test, y_hat_thresh, zero_division=0),
                       "f1_score": f1_score(y_test, y_hat_thresh, zero_division=0)
  }

  curr_score = f1_score(y_test, y_hat_thresh)

  if best_f1_threshold[1] < curr_score:
    best_f1_threshold = (threshold, curr_score)


print(best_f1_threshold)
print(scores[best_f1_threshold[0]])

(np.float64(0.5000000000000001), 0.6511627906976745)
{'precision': 0.5957446808510638, 'recall': 0.717948717948718, 'f1_score': 0.6511627906976745}


## Feature Importance Analysis
Feature importance was evaluated using permutation importance on the final Random Forest model. This approach measures how much model performance degrades when a feature’s values are randomly shuffled, indicating its relative contribution to predictive accuracy.

The most important feature identified were:


*   WAR
*   Runs
*   Slugging Percentage
*   wRC+ (Weighted Runs Created Plus)
*   Batting Average
*   Home Runs
*   Base Running

These results are consistent with expectations, as All-Star selection is largely driven by overall offensive production and total player value. Metrics such as WAR and SLG capture comprehensive performance, while counting stats like Runs and Home Runs reflect visible contributions that often influence selection.


The alignment between model importance and domain knowledge suggests that the model is capturing meaningful performance signals rather than relying on spurious patterns.




In [9]:
from sklearn.inspection import permutation_importance

results = permutation_importance(
    best_model, X_test, y_test, n_repeats=10, random_state=42, scoring='f1'
)

mean_importance = results['importances_mean']
feature_names = best_model.named_steps['transformer'].get_feature_names_out()

feature_importance = zip(feature_names, mean_importance)
feature_importance = sorted(feature_importance, key=lambda x: -x[1])

print(feature_importance[0:7])

[('num_pipe__WAR', np.float64(0.040736471738922087)), ('num_pipe__R', np.float64(0.024189499232110445)), ('num_pipe__SLG', np.float64(0.02081229570117149)), ('num_pipe__wRC+', np.float64(0.007739583551989071)), ('num_pipe__AVG', np.float64(0.006563225742950918)), ('num_pipe__HR', np.float64(0.005038108266562457)), ('num_pipe__BsR', np.float64(0.0047970816233470544))]
